### Trayectoria mínimos cuadrados 3D y trayectoria perturbada por el campo vectorial gravitacional

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl

mpl.rcParams['animation.embed_limit'] = 100

# ============================================
# 1. CONFIGURACIÓN: ESTRELLAS Y CAMPO VECTORIAL
# ============================================

np.random.seed(42)
n_estrellas = 6

# Estrellas distribuidas no linealmente
t_estrellas = np.linspace(0, 10, n_estrellas)
x_est = 2.0 * t_estrellas + np.random.normal(0, 0.6, n_estrellas)
y_est = 1.8 * np.sin(t_estrellas * 0.7) + np.random.normal(0, 0.5, n_estrellas)
z_est = 0.6 * t_estrellas**1.3 + np.random.normal(0, 0.7, n_estrellas)
estrellas = np.column_stack([x_est, y_est, z_est])

masas = np.random.uniform(1.5, 3.0, n_estrellas)

print("Estrellas:")
for i, est in enumerate(estrellas):
    print(f"  {i+1}: ({est[0]:.2f}, {est[1]:.2f}, {est[2]:.2f})  Masa: {masas[i]:.2f}")

# Grid para campo vectorial
n_grid = 6
x_grid = np.linspace(1, 9, n_grid)
y_grid = np.linspace(1, 9, n_grid)
z_grid = np.linspace(1, 9, n_grid)
X_grid, Y_grid, Z_grid = np.meshgrid(x_grid, y_grid, z_grid, indexing='ij')

Fx_grid = np.zeros_like(X_grid)
Fy_grid = np.zeros_like(Y_grid)
Fz_grid = np.zeros_like(Z_grid)

for i in range(n_grid):
    for j in range(n_grid):
        for k in range(n_grid):
            punto = np.array([X_grid[i,j,k], Y_grid[i,j,k], Z_grid[i,j,k]])
            F_total = np.zeros(3)
            for estrella, masa in zip(estrellas, masas):
                r = punto - estrella
                dist = np.linalg.norm(r)
                if dist > 0.8:
                    F_total += -masa * r / (dist**3)
            Fx_grid[i,j,k], Fy_grid[i,j,k], Fz_grid[i,j,k] = F_total

# Normalizar
mags = np.sqrt(Fx_grid**2 + Fy_grid**2 + Fz_grid**2)
Fx_norm = Fx_grid / (mags + 0.1) * 0.8
Fy_norm = Fy_grid / (mags + 0.1) * 0.8
Fz_norm = Fz_grid / (mags + 0.1) * 0.8

# ============================================
# 2. TRAYECTORIA CÚBICA 3D (MÍNIMOS CUADRADOS)
# ============================================

# Matriz de diseño para polinomio cúbico: [1, t, t², t³]
A = np.column_stack([
    np.ones(n_estrellas),
    t_estrellas,
    t_estrellas**2,
    t_estrellas**3
])

# Resolver mínimos cuadrados para cada coordenada
coefs_x, _, _, _ = np.linalg.lstsq(A, estrellas[:, 0], rcond=None)
coefs_y, _, _, _ = np.linalg.lstsq(A, estrellas[:, 1], rcond=None)
coefs_z, _, _, _ = np.linalg.lstsq(A, estrellas[:, 2], rcond=None)

print(f"\nPolinomios cúbicos ajustados:")
print(f"X(t) = {coefs_x[0]:.3f} + {coefs_x[1]:.3f}t + {coefs_x[2]:.3f}t² + {coefs_x[3]:.3f}t³")
print(f"Y(t) = {coefs_y[0]:.3f} + {coefs_y[1]:.3f}t + {coefs_y[2]:.3f}t² + {coefs_y[3]:.3f}t³")
print(f"Z(t) = {coefs_z[0]:.3f} + {coefs_z[1]:.3f}t + {coefs_z[2]:.3f}t² + {coefs_z[3]:.3f}t³")

# Función de trayectoria cúbica
def trayectoria_cubica(t_val):
    T = np.array([1, t_val, t_val**2, t_val**3])
    return np.array([
        np.dot(coefs_x, T),
        np.dot(coefs_y, T),
        np.dot(coefs_z, T)
    ])

# Generar puntos de la curva
t_tray = np.linspace(0, 10, 80)
trayectoria_optima = np.array([trayectoria_cubica(ti) for ti in t_tray])

# ============================================
# 3. TRAYECTORIA PERTURBADA (CAMPO GRAVITACIONAL)
# ============================================

def campo_en_punto(punto):
    F = np.zeros(3)
    for estrella, masa in zip(estrellas, masas):
        r = punto - estrella
        dist = np.linalg.norm(r)
        if dist > 0.3:
            F += -masa * r / (dist**2.5)
    return F

# Simular: el cohete intenta seguir la trayectoria cúbica pero el campo perturba
posicion = trayectoria_optima[0].copy()
# Velocidad tangente a la trayectoria cúbica inicial
velocidad_base = trayectoria_optima[1] - trayectoria_optima[0]
velocidad_base = velocidad_base / np.linalg.norm(velocidad_base) * 0.4

trayectoria_perturbada = [posicion.copy()]
dt = 0.125

for i in range(1, len(t_tray)):
    # Campo gravitacional en posición actual
    F_grav = campo_en_punto(posicion)
    
    # Velocidad deseada (tangente a trayectoria óptima)
    if i < len(t_tray) - 1:
        vel_deseada = trayectoria_optima[i] - trayectoria_optima[i-1]
        vel_deseada = vel_deseada / np.linalg.norm(vel_deseada) * 0.4
    else:
        vel_deseada = velocidad_base
    
    # Perturbación del campo
    velocidad = vel_deseada + 0.15 * F_grav
    
    # Actualizar posición
    posicion = posicion + velocidad * dt
    trayectoria_perturbada.append(posicion.copy())

trayectoria_perturbada = np.array(trayectoria_perturbada)

print(f"\nTrayectoria cúbica: {len(trayectoria_optima)} puntos")
print(f"Trayectoria perturbada: {len(trayectoria_perturbada)} puntos")
print(f"Desviación máxima: {np.max(np.linalg.norm(trayectoria_optima - trayectoria_perturbada, axis=1)):.2f}")

# ============================================
# 4. ANIMACIÓN CON CONTROLES NATIVOS
# ============================================

fig = plt.figure(figsize=(13, 10), dpi=85)
ax = fig.add_subplot(111, projection='3d')

# Estrellas
scatter_estrellas = ax.scatter(estrellas[:, 0], estrellas[:, 1], estrellas[:, 2],
                                c='gold', s=350, marker='*', edgecolors='darkorange', 
                                linewidths=2, label='Estrellas (masas)', zorder=10)

# Campo vectorial
quiver_campo = ax.quiver(X_grid, Y_grid, Z_grid,
                         Fx_norm, Fy_norm, Fz_norm,
                         length=0.9, normalize=False, alpha=0.35, 
                         color='purple', arrow_length_ratio=0.25, linewidth=0.8,
                         label='Campo gravitacional')

# Referencias tenues
ax.plot(trayectoria_optima[:, 0], trayectoria_optima[:, 1], trayectoria_optima[:, 2],
        'lightblue', linewidth=1, alpha=0.2, linestyle='--')
ax.plot(trayectoria_perturbada[:, 0], trayectoria_perturbada[:, 1], trayectoria_perturbada[:, 2],
        'lightgreen', linewidth=1, alpha=0.2, linestyle='--')

# Elementos animados
linea_optima, = ax.plot([], [], [], 'b-', linewidth=3.5, label='Cúbica óptima (grado 3)', zorder=5)
linea_perturbada, = ax.plot([], [], [], 'g-', linewidth=3.5, label='Perturbada (campo)', zorder=5)

nave_optima, = ax.plot([], [], [], 'b^', markersize=12, 
                       markeredgecolor='navy', markeredgewidth=2.5, zorder=8)
nave_perturbada, = ax.plot([], [], [], 'gs', markersize=12, 
                           markeredgecolor='darkgreen', markeredgewidth=2.5, zorder=8)

desviacion, = ax.plot([], [], [], 'r--', alpha=0.6, linewidth=2, label='Desviación', zorder=6)

# Ejes
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_zlim(0, 10)
ax.set_xlabel('X (años luz)', fontsize=11)
ax.set_ylabel('Y (años luz)', fontsize=11)
ax.set_zlabel('Z (años luz)', fontsize=11)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)

n_frames = len(t_tray)

def init():
    linea_optima.set_data([], [])
    linea_optima.set_3d_properties([])
    linea_perturbada.set_data([], [])
    linea_perturbada.set_3d_properties([])
    nave_optima.set_data([], [])
    nave_optima.set_3d_properties([])
    nave_perturbada.set_data([], [])
    nave_perturbada.set_3d_properties([])
    desviacion.set_data([], [])
    desviacion.set_3d_properties([])
    return linea_optima, linea_perturbada, nave_optima, nave_perturbada, desviacion

def animate(frame):
    idx = frame
    
    # Trayectoria cúbica óptima (AZUL)
    linea_optima.set_data(trayectoria_optima[:idx+1, 0], trayectoria_optima[:idx+1, 1])
    linea_optima.set_3d_properties(trayectoria_optima[:idx+1, 2])
    
    # Trayectoria perturbada (VERDE)
    linea_perturbada.set_data(trayectoria_perturbada[:idx+1, 0], trayectoria_perturbada[:idx+1, 1])
    linea_perturbada.set_3d_properties(trayectoria_perturbada[:idx+1, 2])
    
    # Naves
    nave_optima.set_data([trayectoria_optima[idx, 0]], [trayectoria_optima[idx, 1]])
    nave_optima.set_3d_properties([trayectoria_optima[idx, 2]])
    
    nave_perturbada.set_data([trayectoria_perturbada[idx, 0]], [trayectoria_perturbada[idx, 1]])
    nave_perturbada.set_3d_properties([trayectoria_perturbada[idx, 2]])
    
    # Línea de desviación
    desviacion.set_data(
        [trayectoria_optima[idx, 0], trayectoria_perturbada[idx, 0]],
        [trayectoria_optima[idx, 1], trayectoria_perturbada[idx, 1]]
    )
    desviacion.set_3d_properties(
        [trayectoria_optima[idx, 2], trayectoria_perturbada[idx, 2]]
    )
    
    # Título
    dist = np.linalg.norm(trayectoria_optima[idx] - trayectoria_perturbada[idx])
    progreso = (idx / n_frames) * 100
    ax.set_title(f'Trayectoria Cúbica Grado 3 + Campo Gravitacional | {progreso:.0f}% | Desv: {dist:.2f} a.l.', 
                 fontsize=11, fontweight='bold')
    
    # Rotación
    ax.view_init(elev=20, azim=45 + idx * 0.5)
    
    return linea_optima, linea_perturbada, nave_optima, nave_perturbada, desviacion

anim = FuncAnimation(fig, animate, init_func=init,
                     frames=n_frames, interval=120, blit=False, repeat=True)

plt.close(fig)
html_anim = anim.to_jshtml()
display(HTML(html_anim))